# 12 · Сегментация / Image segmentation
**RU.** Реальные фотографии Oxford-IIIT Pet (Parkhi et al., 2012): наблюдение — изображение животного, признаки — RGB-пиксели, цель — маска foreground/background. [Официальный источник](https://www.robots.ox.ac.uk/~vgg/data/pets/). Trimap: 1 — объект, 2 — фон, 3 — неоднозначная граница; значение 3 исключаем из метрик. Это учебный бинарный протокол, не классификация пород и не официальная benchmark-оценка. Данные доступны для исследований; соблюдайте условия исходных сайтов изображений, указанные авторами. Изображения не распространяются в репозитории курса.
**EN.** Real Oxford-IIIT Pet photographs: each observation is an animal image, RGB pixels are features, and a foreground/background mask is the target. Official trimaps use 1 for foreground, 2 for background, 3 for unclassified boundary. Exclude label 3 from metrics. This binary teaching protocol is neither breed classification nor an official benchmark score. The dataset is released for research; original image website terms still apply. Images are not redistributed in this repository.

**RU.** CPU; NumPy, Pillow, OpenCV, matplotlib. Первый запуск скачивает около 800 МБ, повторный использует кэш. Зафиксированы 40 train, 8 dev и 8 test: равное число кошек и собак, seed 1212. Train/dev — из официального trainval, test — из test. Идентификаторы непересекаются; сведения об отдельных животных отсутствуют, независимость по животным не заявляется. Маски не используются для выбора рамки. Параметры выбираем только по dev; test открываем после фиксации правила.
**EN.** CPU; NumPy, Pillow, OpenCV, matplotlib. Initial download is about 800 MB; subsequent runs reuse the cache. Fixed 40 train, 8 dev and 8 test images, equally many cats and dogs, seed 1212. Train/dev come from official trainval; test comes from official test. IDs are disjoint, but animal identity independence is not established. Masks do not determine input rectangles. Tune on dev, freeze the rule, then evaluate test.


In [ ]:
from pathlib import Path  # RU: управляем кэшем; EN: manage the cache.
import json, hashlib, tarfile, urllib.request  # RU: читаем и проверяем архивы; EN: read and verify archives.
import numpy as np  # RU: обрабатываем маски; EN: process masks.
from PIL import Image  # RU: декодируем изображения; EN: decode images.
import cv2  # RU: используем GrabCut; EN: use GrabCut.
import matplotlib.pyplot as plt  # RU: показываем фотографии; EN: display photographs.
data = next((p for p in [Path('build/data'), Path('../build/data'), Path('data')] if (p/'images.tar.gz').exists()), Path('data'))  # RU: находим или создаём кэш; EN: locate or create the cache.
data.mkdir(exist_ok=True, parents=True)  # RU: создаём каталог данных; EN: create the data directory.
resources = {'images.tar.gz':'5c4f3ee8e5d25df40f4fd59a7f44e54c', 'annotations.tar.gz':'95a8c909bbe2e81eed6a22bccdf3f68f'}  # RU: фиксируем контрольные суммы torchvision; EN: pin torchvision checksums.
for filename, checksum in resources.items():  # RU: проверяем оба архива; EN: check both archives.
    path = data/filename  # RU: формируем путь архива; EN: construct archive path.
    if not path.exists():  # RU: избегаем повторной загрузки; EN: avoid repeat downloads.
        urllib.request.urlretrieve('https://www.robots.ox.ac.uk/~vgg/data/pets/data/'+filename, path)  # RU: загружаем официальный архив; EN: download the official archive.
    assert hashlib.md5(path.read_bytes()).hexdigest() == checksum  # RU: проверяем целостность; EN: verify integrity.
PARTS = {'train': ['Egyptian_Mau_155', 'Bombay_175', 'Egyptian_Mau_190', 'Russian_Blue_150', 'Abyssinian_1', 'Maine_Coon_141', 'Bengal_183', 'Egyptian_Mau_132', 'Abyssinian_114', 'Sphynx_200', 'Ragdoll_159', 'Abyssinian_197', 'Ragdoll_125', 'Egyptian_Mau_147', 'Siamese_155', 'Sphynx_115', 'Maine_Coon_150', 'Bombay_107', 'Bengal_136', 'Birman_155', 'german_shorthaired_180', 'keeshond_184', 'english_cocker_spaniel_103', 'staffordshire_bull_terrier_131', 'shiba_inu_106', 'great_pyrenees_142', 'chihuahua_136', 'german_shorthaired_181', 'german_shorthaired_113', 'wheaten_terrier_112', 'staffordshire_bull_terrier_159', 'chihuahua_187', 'yorkshire_terrier_186', 'yorkshire_terrier_118', 'japanese_chin_108', 'basset_hound_179', 'japanese_chin_120', 'wheaten_terrier_142', 'keeshond_163', 'german_shorthaired_13'], 'dev': ['British_Shorthair_182', 'Bengal_104', 'Egyptian_Mau_107', 'Ragdoll_143', 'shiba_inu_142', 'keeshond_158', 'english_setter_151', 'basset_hound_147'], 'test': ['Egyptian_Mau_213', 'Abyssinian_43', 'Russian_Blue_207', 'Maine_Coon_268', 'wheaten_terrier_69', 'staffordshire_bull_terrier_40', 'chihuahua_191', 'beagle_61']}  # RU: сохраняем идентификаторы разбиения; EN: preserve split IDs.
assert not (set(PARTS['train']) & set(PARTS['dev']) or set(PARTS['train']) & set(PARTS['test']) or set(PARTS['dev']) & set(PARTS['test']))  # RU: исключаем пересечения; EN: reject overlapping IDs.
images, trimaps = {}, {}  # RU: создаём коллекции наблюдений и меток; EN: create observation and label collections.
with tarfile.open(data/'images.tar.gz') as archive:  # RU: читаем без распаковки путей; EN: read without extracting paths.
    for name in sum(PARTS.values(), []):  # RU: загружаем фиксированную подвыборку; EN: load the fixed subset.
        images[name] = np.asarray(Image.open(archive.extractfile('images/'+name+'.jpg')).convert('RGB').resize((128,128), Image.Resampling.BILINEAR))  # RU: масштабируем RGB; EN: resize RGB images.
with tarfile.open(data/'annotations.tar.gz') as archive:  # RU: открываем разметку; EN: open annotations.
    for name in images:  # RU: сопоставляем по имени, не позиции; EN: match by ID rather than position.
        trimaps[name] = np.asarray(Image.open(archive.extractfile('annotations/trimaps/'+name+'.png')).resize((128,128), Image.Resampling.NEAREST))  # RU: сохраняем дискретные метки; EN: preserve discrete labels.
name = PARTS['dev'][0]  # RU: выбираем пример dev; EN: select a development example.
fig, axes = plt.subplots(1,2,figsize=(8,4))  # RU: сопоставляем изображение и разметку; EN: compare image and annotation.
axes[0].imshow(images[name]); axes[1].imshow(trimaps[name],vmin=1,vmax=3)  # RU: показываем исходные данные; EN: display source data.
plt.show()  # RU: выводим рисунок; EN: render the figure.


## 1 · Trimap → цель и valid / Target and valid mask
**RU · 5–10 минут.** Реализуйте `decode_trimap(mask)`: два bool-массива той же формы, target=(mask==1), valid=(mask!=3). Запретите значения вне {1,2,3} через ValueError. Покажите target и valid для первого dev-изображения. Объясните, почему значение 3 нельзя автоматически считать фоном.
**EN · 5–10 minutes.** Implement `decode_trimap(mask)`, returning boolean target=(mask==1) and valid=(mask!=3), same shape as input. Raise ValueError for labels outside {1,2,3}. Display both arrays for the first dev image and explain why label 3 is not automatically background.

**RU.** Перед кодом сформулируйте вход, выход и случай, когда оценка не определена. **EN.** State the input, output, and undefined case before coding.

In [ ]:
print('Image:', name, 'shape:', images[name].shape, 'labels:', np.unique(trimaps[name]))  # RU: проверяем наблюдение перед вычислением; EN: inspect the observation before computing.

In [ ]:
def decode_trimap(mask):  # RU: разделяем цель и область оценки; EN: separate target and evaluation region.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте контракт; EN: implement the contract.

In [ ]:
target, valid = decode_trimap(trimaps[name])  # RU: декодируем реальную маску; EN: decode a real mask.
assert target.dtype == bool and valid.dtype == bool  # RU: проверяем тип; EN: check dtype.
assert np.array_equal(target,trimaps[name]==1) and np.array_equal(valid,trimaps[name]!=3)  # RU: сверяем семантику; EN: verify semantics.


## 2 · IoU и Dice / IoU and Dice
**RU · 5–10 минут.** Реализуйте `mask_metrics(pred,target,valid)`: словарь iou/dice по valid-пикселям. IoU=intersection/union, Dice=2intersection/(pred_area+target_area). Несовпадение форм — ValueError; нулевой знаменатель — NaN. Сравните для настоящей маски две простые гипотезы: всё foreground и всё background. Не делайте вывод по одной pixel accuracy.
**EN · 5–10 minutes.** Implement `mask_metrics(pred,target,valid)` returning iou/dice over valid pixels. Reject mismatched shapes; return NaN for a zero denominator. Compare all-foreground and all-background predictions against the real annotation, explaining why pixel accuracy alone can mislead.

**RU.** Перед кодом сформулируйте вход, выход и случай, когда оценка не определена. **EN.** State the input, output, and undefined case before coding.

In [ ]:
print('Image:', name, 'shape:', images[name].shape, 'labels:', np.unique(trimaps[name]))  # RU: проверяем наблюдение перед вычислением; EN: inspect the observation before computing.

In [ ]:
def mask_metrics(pred, target, valid):  # RU: оцениваем только размеченную область; EN: evaluate the labeled region only.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте контракт; EN: implement the contract.

In [ ]:
perfect = mask_metrics(target,target,valid)  # RU: проверяем идеальное совпадение; EN: check perfect overlap.
assert np.isclose(perfect['iou'],1) and np.isclose(perfect['dice'],1)  # RU: обе метрики равны единице; EN: both metrics equal one.
base = mask_metrics(np.ones_like(target),target,valid)  # RU: оцениваем тривиальный foreground; EN: evaluate all-foreground baseline.
assert np.isclose(base['iou'],np.sum(target & valid)/valid.sum())  # RU: сверяем независимый расчёт; EN: verify independent calculation.


## 3 · GrabCut и рамка / GrabCut and rectangle
**RU · 5–10 минут.** Реализуйте `grabcut_mask(rgb,margin)`: margin — доля отступа с каждой стороны, 0<margin<0.5. dx=max(1,round(width*margin)), аналогично dy; рамка (dx,dy,w-2dx,h-2dy). Seed 1212, три итерации, GC_INIT_WITH_RECT. Возвращайте bool: состояния GC_FGD или GC_PR_FGD. Значения 0,1,2,3 GrabCut имеют другой смысл, чем trimap! Показать результат и вычислить метрики для dev. Рамку не строить из ground truth.
**EN · 5–10 minutes.** Implement `grabcut_mask(rgb,margin)` using the specified relative inset, seed 1212, three iterations and rectangle initialization. Return true for definite or probable foreground. Reject margins outside (0,0.5). Display and evaluate a dev result. GrabCut states differ from dataset trimap labels. Do not derive the rectangle from ground truth.

**RU.** Перед кодом сформулируйте вход, выход и случай, когда оценка не определена. **EN.** State the input, output, and undefined case before coding.

In [ ]:
print('Image:', name, 'shape:', images[name].shape, 'labels:', np.unique(trimaps[name]))  # RU: проверяем наблюдение перед вычислением; EN: inspect the observation before computing.

In [ ]:
def grabcut_mask(rgb, margin):  # RU: сегментируем с фиксированной рамкой; EN: segment with a fixed rectangle.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте контракт; EN: implement the contract.

In [ ]:
prediction = grabcut_mask(images[name],.1)  # RU: запускаем на настоящем изображении; EN: run on a real image.
assert prediction.shape==target.shape and prediction.dtype==bool  # RU: проверяем контракт маски; EN: check the mask contract.
assert not prediction[0].any() and not prediction[-1].any()  # RU: вне рамки фиксируется фон; EN: outside the rectangle is background.
print(mask_metrics(prediction,target,valid))  # RU: печатаем измеренные метрики; EN: print measured metrics.


## 4 · Выбор на dev / Development selection
**RU · 5–10 минут.** Реализуйте `choose_margin(records)`: список словарей margin/mean_iou, максимизировать конечный mean_iou, при равенстве брать меньший margin; если конечных оценок нет — None. Сравните .03,.10,.20 по среднему IoU восьми dev-изображений; не усредняйте NaN как ноль. Зафиксируйте margin, затем один раз оцените восемь test. Покажите худший test-пример, перечислите ограничения рамки и маленькой выборки.
**EN · 5–10 minutes.** Implement `choose_margin(records)`: maximize finite mean_iou, break ties by smaller margin, return None if none are finite. Compare .03,.10,.20 on eight dev images, excluding undefined scores. Freeze the choice and evaluate eight test images once. Display the worst case and discuss rectangle and sample-size limitations.

**RU.** Перед кодом сформулируйте вход, выход и случай, когда оценка не определена. **EN.** State the input, output, and undefined case before coding.

In [ ]:
print('Image:', name, 'shape:', images[name].shape, 'labels:', np.unique(trimaps[name]))  # RU: проверяем наблюдение перед вычислением; EN: inspect the observation before computing.

In [ ]:
def choose_margin(records):  # RU: выбираем по dev без доступа к test; EN: select on dev without test access.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте контракт; EN: implement the contract.

In [ ]:
records = []  # RU: собираем результаты кандидатов; EN: collect candidate results.
for margin in [.03,.10,.20]:  # RU: используем заранее заданный набор; EN: use predeclared candidates.
    values = [mask_metrics(grabcut_mask(images[n],margin),*decode_trimap(trimaps[n]))['iou'] for n in PARTS['dev']]  # RU: оцениваем только dev; EN: evaluate dev only.
    records.append({'margin':margin,'mean_iou':float(np.nanmean(values))})  # RU: усредняем по изображениям; EN: average across images.
chosen = choose_margin(records)  # RU: фиксируем выбранный параметр; EN: freeze the selected parameter.
assert chosen == sorted(records,key=lambda r:(-r['mean_iou'],r['margin']))[0]['margin']  # RU: сверяем правило выбора; EN: verify selection rule.
assert choose_margin([]) is None  # RU: проверяем отсутствие кандидатов; EN: check absent candidates.
test_scores = [mask_metrics(grabcut_mask(images[n],chosen),*decode_trimap(trimaps[n]))['iou'] for n in PARTS['test']]  # RU: проводим итоговую проверку; EN: perform final evaluation.
print('Frozen margin:',chosen,'Test mean IoU:',float(np.nanmean(test_scores)))  # RU: показываем результат без перенастройки; EN: report without retuning.
